In [2]:
"""
클래스 중복 라벨 시각 판별 도구

원본 폴더의 각 클래스에서
  - 평균 이미지(대표 색/형태) 1장
  - 무작위 대표 샘플 N장
을 뽑아 한 장의 격자 PNG로 만든다. 이 격자 하나만 보면 어떤 클래스끼리
시각적으로 같은지(중복 라벨) 폴더를 일일이 열지 않고 판별할 수 있다.

추가로, 평균 이미지 간 유사도(코사인)를 계산해 '중복 의심 쌍 TOP'을 표로 출력한다.
=> 사람 눈(격자) + 수치(유사도) 두 근거로 병합 대상을 확정.
"""

import os
import random
import itertools
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt

# ---- 한글 폰트 (Windows) ----
try:
    matplotlib.rc('font', family='Malgun Gothic')
    matplotlib.rcParams['axes.unicode_minus'] = False
except Exception:
    pass


# ==========================================================
# 설정
# ==========================================================
PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE

# ★ 검토할 원본 폴더 (클래스별 하위폴더). 스크린샷의 그 폴더 경로로 수정
ORIGINAL_DIR = r"C:\Users\user\Desktop\졸작_A모델_학습용_크롭데이터"   # ← 실제 경로로 수정

OUT_GRID = os.path.join(DATA_DIR, "class_grid.png")
N_SAMPLES = 5          # 클래스당 대표 샘플 수
THUMB = 96             # 썸네일 한 변 픽셀
MEAN_LIMIT = 150       # 평균 이미지 계산에 쓸 최대 장수
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')


def list_images(d):
    return [f for f in os.listdir(d) if f.lower().endswith(IMG_EXTS)]


def load_thumb(path, size=THUMB):
    return np.asarray(Image.open(path).convert('RGB').resize((size, size)), dtype=np.float32)


def build(root, out_png):
    classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    print(f"클래스 {len(classes)}개 발견")

    means = {}
    counts = {}
    fig, axes = plt.subplots(len(classes), N_SAMPLES + 1,
                             figsize=((N_SAMPLES + 1) * 1.6, len(classes) * 1.6))
    if len(classes) == 1:
        axes = axes[None, :]

    random.seed(0)
    for r, c in enumerate(classes):
        cdir = os.path.join(root, c)
        files = list_images(cdir)
        counts[c] = len(files)

        # 평균 이미지
        acc, cnt = None, 0
        for f in files[:MEAN_LIMIT]:
            try:
                im = load_thumb(os.path.join(cdir, f))
                acc = im if acc is None else acc + im
                cnt += 1
            except Exception:
                pass
        mean_img = (acc / cnt) if cnt else np.zeros((THUMB, THUMB, 3), np.float32)
        means[c] = mean_img.flatten()

        axes[r, 0].imshow(mean_img.astype('uint8'))
        axes[r, 0].set_ylabel(f"{c}\n({counts[c]})", rotation=0, ha='right', va='center', fontsize=8)
        axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
        if r == 0:
            axes[r, 0].set_title('평균', fontsize=9)

        # 대표 샘플
        samp = random.sample(files, min(N_SAMPLES, len(files)))
        for j in range(N_SAMPLES):
            axes[r, j + 1].set_xticks([]); axes[r, j + 1].set_yticks([])
            if r == 0:
                axes[r, j + 1].set_title(f'예시{j+1}', fontsize=9)
            if j < len(samp):
                try:
                    axes[r, j + 1].imshow(load_thumb(os.path.join(cdir, samp[j])).astype('uint8'))
                except Exception:
                    pass

    plt.tight_layout()
    plt.savefig(out_png, dpi=120)
    plt.close()
    print(f"격자 이미지 저장: {out_png}")

    # ---- 평균 이미지 코사인 유사도로 중복 의심 쌍 ----
    print("\n[중복 의심 쌍 TOP15] (평균 이미지 유사도 높은 순)")
    print("  ※ 유사도가 높다고 반드시 중복은 아님 — 격자로 눈 확인 필수")
    names = list(means.keys())
    vecs = np.stack([means[n] for n in names])
    vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8)
    sims = []
    for i, j in itertools.combinations(range(len(names)), 2):
        sims.append((float(vecs[i] @ vecs[j]), names[i], names[j]))
    sims.sort(reverse=True)
    for s, a, b in sims[:15]:
        print(f"  {s:.3f}  {a:<18} ↔ {b}")

    print("\n[클래스별 장수]")
    for c in sorted(counts, key=lambda x: counts[x]):
        print(f"  {c:<20} {counts[c]:>6}장")


if __name__ == "__main__":
    if not os.path.isdir(ORIGINAL_DIR):
        raise FileNotFoundError(f"원본 폴더 없음: {ORIGINAL_DIR}  ← ORIGINAL_DIR 경로를 수정하세요")
    build(ORIGINAL_DIR, OUT_GRID)
    print("\n완료. class_grid.png를 열어 어떤 클래스끼리 같은 옷인지 확인하세요.")

클래스 22개 발견
격자 이미지 저장: C:\Users\user\Desktop\졸작_최종_파이프라인\class_grid.png

[중복 의심 쌍 TOP15] (평균 이미지 유사도 높은 순)
  ※ 유사도가 높다고 반드시 중복은 아님 — 격자로 눈 확인 필수
  0.999  2_padding          ↔ 3_training_zipup
  0.998  10_outer_jacket    ↔ 1_coat
  0.998  10_outer_jacket    ↔ 무스탕
  0.998  1_coat             ↔ 무스탕
  0.997  4_sweatshirt       ↔ 후드집업
  0.997  4_sweatshirt       ↔ 7_shirt
  0.997  가디건                ↔ 후드집업
  0.997  6_shortsleeve      ↔ 7_shirt
  0.997  8_knit             ↔ 블레이저
  0.997  10_outer_jacket    ↔ 후드집업
  0.997  2_padding          ↔ 5_hoodie
  0.997  3_training_zipup   ↔ 5_hoodie
  0.996  9_jacket           ↔ 가디건
  0.996  5_hoodie           ↔ 9_jacket
  0.996  블레이저               ↔ 청자켓

[클래스별 장수]
  코튼팬츠                     28장
  트레이닝팬츠                  102장
  3_training_zipup        116장
  반바지                     139장
  4_sweatshirt            240장
  청바지                     341장
  8_knit                  391장
  2_padding               395장
  6_shortsleeve           568장
  1_coat     

In [ ]:
"""
Phase 1 데모 (통합판) — 사전학습 모델만으로 end-to-end 관통

포함 기능:
  0) 원본 데이터셋에서 카테고리별 N장씩 뽑아 demo_garments 폴더 자동 생성
     + 착용샷 후보(검출 옷 2개 이상) 자동 탐색
  1) 옷 검출/세그 (YOLOv8-seg, DeepFashion2 사전학습)
  2) 마스크 크롭 (배경 흰색 처리 → FashionSigLIP 최적 조건)
  3) FashionSigLIP 이미지/텍스트 임베딩
  4) 텍스트 쿼리 → 코사인 유사도 추천

MODE:
  "sample" : demo_garments 생성 + 착용샷 후보 탐색만 하고 종료
  "demo"   : 이미 만들어진 demo_garments로 검출/추천 시연만
  "all"    : 샘플 생성 후 이어서 시연 (기본)

설치:
  pip install ultralytics transformers torch pillow huggingface_hub numpy
"""

import os
import glob
import random
import shutil
import numpy as np
import torch
from PIL import Image

from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from transformers import AutoModel, AutoProcessor


# ==========================================================
# 설정
# ==========================================================
MODE = "all"     # "sample" | "demo" | "all"

# ★ 원본 데이터셋 (클래스별 하위폴더). 스크린샷의 그 폴더
ORIGINAL_DIR = r"D:\백업 시작\졸작_A모델_학습용_크롤데이터"   # ← 수정

# 데모용 산출 폴더/파일 (자동 생성됨)
WORK_DIR = r"C:\Users\user\Desktop\phase1_demo"
GARMENT_DIR = os.path.join(WORK_DIR, "demo_garments")       # 추천 후보 옷 DB
QUERY_IMAGE = os.path.join(WORK_DIR, "query.jpg")           # 검출 테스트 착용샷

PER_CLASS = 8            # 카테고리별 샘플 수 (추천 다양성 확보용)
FIND_WEARING_SHOTS = 5   # 착용샷 후보 최대 개수

# 모델
DET_REPO, DET_FILE = "Bingsu/adetailer", "deepfashion2_yolov8s-seg.pt"
EMB_MODEL = "Marqo/marqo-fashionSigLIP"   # 폴백: "Marqo/marqo-fashionCLIP"
DET_CONF = 0.25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DF2_CLASSES = [
    "short_sleeved_shirt", "long_sleeved_shirt", "short_sleeved_outwear",
    "long_sleeved_outwear", "vest", "sling", "shorts", "trousers", "skirt",
    "short_sleeved_dress", "long_sleeved_dress", "vest_dress", "sling_dress",
]
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# 테스트할 텍스트 쿼리 (FashionSigLIP는 영어 학습 → 영어 권장)
QUERIES = [
    "cool tone street style",
    "minimal formal look",
    "warm cozy winter outfit",
    "casual denim style",
]


# ==========================================================
# [0] 샘플 폴더 생성 + 착용샷 탐색
# ==========================================================
def build_demo_sample(src_root, out_dir, per_class=8, seed=42):
    random.seed(seed)
    if not os.path.isdir(src_root):
        raise FileNotFoundError(f"원본 폴더 없음: {src_root}  ← ORIGINAL_DIR 수정")
    classes = sorted([d for d in os.listdir(src_root) if os.path.isdir(os.path.join(src_root, d))])
    os.makedirs(out_dir, exist_ok=True)
    print(f"[0] demo_garments 생성 (카테고리 {len(classes)}개, 클래스당 {per_class}장)")
    total = 0
    for c in classes:
        cdir = os.path.join(src_root, c)
        files = [f for f in os.listdir(cdir) if f.lower().endswith(IMG_EXTS)]
        pick = random.sample(files, min(per_class, len(files)))
        for f in pick:
            # 카테고리를 파일명 프리픽스로 남겨 추천 결과에서 식별 가능
            shutil.copy2(os.path.join(cdir, f), os.path.join(out_dir, f"{c}__{f}"))
        total += len(pick)
    print(f"    → {out_dir} 에 총 {total}장 복사")
    return classes


def find_wearing_shots(detector, src_root, max_shots=5, scan_limit=400):
    """검출 결과 옷이 2개 이상인 이미지(=착용샷 후보)를 찾아 반환."""
    print(f"[0] 착용샷 후보 탐색 (검출 옷 2개 이상, 최대 {scan_limit}장 스캔)...")
    random.seed(1)
    all_files = []
    for c in os.listdir(src_root):
        cdir = os.path.join(src_root, c)
        if os.path.isdir(cdir):
            all_files += [os.path.join(cdir, f) for f in os.listdir(cdir)
                          if f.lower().endswith(IMG_EXTS)]
    random.shuffle(all_files)
    found = []
    for f in all_files[:scan_limit]:
        try:
            r = detector.predict(f, conf=DET_CONF, verbose=False)[0]
            n = 0 if r.boxes is None else len(r.boxes)
            if n >= 2:
                found.append((f, n))
                print(f"    후보: {os.path.basename(f)} (옷 {n}개)")
                if len(found) >= max_shots:
                    break
        except Exception:
            pass
    if not found:
        print("    착용샷 후보 없음 — 데이터가 쇼핑몰컷 위주일 수 있음")
    return found


# ==========================================================
# 모델 로딩
# ==========================================================
def load_detector():
    print("검출 모델 로딩...")
    return YOLO(hf_hub_download(DET_REPO, DET_FILE))


def load_embedder():
    print(f"임베딩 모델 로딩: {EMB_MODEL}")
    model = AutoModel.from_pretrained(EMB_MODEL, trust_remote_code=True).to(DEVICE).eval()
    processor = AutoProcessor.from_pretrained(EMB_MODEL, trust_remote_code=True)
    return model, processor


# ==========================================================
# [1]+[2] 검출 & 마스크 크롭
# ==========================================================
def crop_with_mask(img_rgb, mask, bbox, bg=(255, 255, 255)):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    if mask is not None:
        m3 = mask[..., None].astype(bool)
        img_rgb = np.where(m3, img_rgb, np.array(bg, dtype=np.uint8))
    return img_rgb[y1:y2, x1:x2]


def detect_and_crop(detector, image_path):
    img = np.asarray(Image.open(image_path).convert("RGB"))
    H, W = img.shape[:2]
    r = detector.predict(image_path, conf=DET_CONF, verbose=False)[0]
    crops = []
    if r.boxes is None or len(r.boxes) == 0:
        return crops
    boxes = r.boxes.xyxy.cpu().numpy()
    clss = r.boxes.cls.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    masks = r.masks.data.cpu().numpy() if r.masks is not None else None
    for i in range(len(boxes)):
        mask_i = None
        if masks is not None:
            m = masks[i]
            if m.shape != (H, W):
                m_img = Image.fromarray((m * 255).astype(np.uint8)).resize((W, H))
                m = (np.asarray(m_img) > 127).astype(np.uint8)
            mask_i = m
        crop = crop_with_mask(img, mask_i, boxes[i])
        if crop.size == 0:
            continue
        cat = DF2_CLASSES[clss[i]] if clss[i] < len(DF2_CLASSES) else str(clss[i])
        crops.append((Image.fromarray(crop), cat, float(confs[i])))
    return crops


# ==========================================================
# [3] 임베딩
# ==========================================================
@torch.no_grad()
def embed_images(model, processor, pil_images):
    inputs = processor(images=pil_images, return_tensors="pt").to(DEVICE)
    feats = model.get_image_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()


@torch.no_grad()
def embed_texts(model, processor, texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(DEVICE)
    feats = model.get_text_features(**inputs)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()


# ==========================================================
# 옷 DB 구축 + 추천
# ==========================================================
def build_garment_db(detector, embedder, processor, garment_dir):
    files = []
    for e in IMG_EXTS:
        files += glob.glob(os.path.join(garment_dir, f"*{e}"))
    print(f"[DB] 옷 이미지 {len(files)}장 임베딩 중...")
    db = []
    for f in files:
        crops = detect_and_crop(detector, f)
        if not crops:   # 검출 실패 → 이미지 전체를 옷으로 (쇼핑몰컷 대응)
            crops = [(Image.open(f).convert("RGB"), "whole_image", 1.0)]
        embs = embed_images(embedder, processor, [c[0] for c in crops])
        for (crop, cat, conf), emb in zip(crops, embs):
            db.append({"emb": emb, "category": cat, "conf": conf, "source": f})
    print(f"[DB] 총 옷 아이템 {len(db)}개")
    return db


def recommend(embedder, processor, db, query_text, top_k=5, category_filter=None):
    tvec = embed_texts(embedder, processor, [query_text])[0]
    items = db if category_filter is None else [d for d in db if category_filter in d["category"]]
    if not items:
        return []
    sims = np.stack([d["emb"] for d in items]) @ tvec
    order = np.argsort(-sims)[:top_k]
    return [(items[i], float(sims[i])) for i in order]


# ==========================================================
# 메인
# ==========================================================
if __name__ == "__main__":
    os.makedirs(WORK_DIR, exist_ok=True)

    # ---- 샘플 생성 단계 ----
    if MODE in ("sample", "all"):
        build_demo_sample(ORIGINAL_DIR, GARMENT_DIR, PER_CLASS)
        # 착용샷 후보는 검출 모델이 필요하므로 여기서 로딩
        det_for_scan = load_detector()
        shots = find_wearing_shots(det_for_scan, ORIGINAL_DIR, FIND_WEARING_SHOTS)
        if shots and not os.path.exists(QUERY_IMAGE):
            shutil.copy2(shots[0][0], QUERY_IMAGE)
            print(f"    → 대표 착용샷을 query.jpg로 저장: {os.path.basename(shots[0][0])}")
        if MODE == "sample":
            print("\n[sample 모드] demo_garments/ 와 착용샷 후보 확인 후, "
                  "MODE='demo'로 다시 실행하세요.")
            raise SystemExit(0)
        detector = det_for_scan
    else:
        detector = load_detector()

    # ---- 시연 단계 ----
    embedder, processor = load_embedder()

    # 검출 시연
    if os.path.exists(QUERY_IMAGE):
        print(f"\n[검출 시연] {os.path.basename(QUERY_IMAGE)}")
        crops = detect_and_crop(detector, QUERY_IMAGE)
        print(f"  검출된 옷 {len(crops)}개:")
        for i, (crop, cat, conf) in enumerate(crops):
            print(f"    {i}: {cat} (conf {conf:.2f}, 크기 {crop.size})")
            crop.save(os.path.join(WORK_DIR, f"crop_{i}_{cat}.jpg"))
        print(f"  크롭 이미지들을 {WORK_DIR}에 저장(눈으로 확인 가능)")
    else:
        print("\n[검출 시연] query.jpg 없음 — 착용샷을 직접 넣거나 데이터에 착용샷이 없는 경우")

    # 추천 시연
    if os.path.isdir(GARMENT_DIR):
        db = build_garment_db(detector, embedder, processor, GARMENT_DIR)
        for q in QUERIES:
            print(f"\n[추천] '{q}'")
            for rank, (item, sim) in enumerate(recommend(embedder, processor, db, q, 3), 1):
                print(f"  {rank}. sim={sim:.3f} | {item['category']:<20} | "
                      f"{os.path.basename(item['source'])}")

    print("\nPhase 1 데모 완료.")
    print("확인 포인트: ① 착용샷에서 옷이 여러 개 분리 검출되는가 "
          "② 추천이 쿼리 분위기와 맞는가")

OSError: [WinError 4551] 애플리케이션 제어 정책에서 이 파일을 차단했습니다. Error loading "C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\lib\asmjit.dll" or one of its dependencies.

: 